In [1]:
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DB_PATH = os.path.join(BASE_DIR, "2_DATABASE", "eko_cards.db")
PRICES_PATH = os.path.join(BASE_DIR, "3_PROJECT_DATA", "prices.xlsx")

In [2]:
def merge_tables():
    import sqlite3
    import pandas as pd

    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT
        t.*,
        c.company,
        p.margin,
        p.discount,
        p.final_price AS base_price,

        CASE
            WHEN COALESCE(p.discount, 0) > 0 THEN
                CAST(
                    (
                        CASE
                            WHEN (t.price - COALESCE(p.discount, 0)) < 0 THEN 0
                            ELSE (t.price - COALESCE(p.discount, 0))
                        END
                    ) * 100 + 0.5
                AS INTEGER) / 100.0

            WHEN p.final_price IS NOT NULL THEN
                CAST(p.final_price * 100 + 0.5 AS INTEGER) / 100.0

            ELSE
                CAST(t.price * 100 + 0.5 AS INTEGER) / 100.0
        END AS final_price

    FROM transactions t

    LEFT JOIN cards c
        ON t.card_number = c.card_number

    LEFT JOIN prices p
        ON p.eik = c.eik
        AND UPPER(p.product) = UPPER(t.material)
        AND p.date = (
            SELECT MAX(p2.date)
            FROM prices p2
            WHERE p2.eik = c.eik
              AND UPPER(p2.product) = UPPER(t.material)
              AND p2.date <= t.date
        )
        AND p.rowid = (
            SELECT p3.rowid
            FROM prices p3
            WHERE p3.eik = c.eik
              AND UPPER(p3.product) = UPPER(t.material)
              AND p3.date = (
                  SELECT MAX(p4.date)
                  FROM prices p4
                  WHERE p4.eik = c.eik
                    AND UPPER(p4.product) = UPPER(t.material)
                    AND p4.date <= t.date
              )
            ORDER BY p3.rowid DESC
            LIMIT 1
        )
    """

    data = pd.read_sql(query, conn)
    conn.close()

    return data

In [3]:
def test_row_count():
    df = merge_tables()

    import sqlite3
    conn = sqlite3.connect(DB_PATH)
    total = conn.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
    conn.close()

    print("Transactions:", total)
    print("Result rows:", len(df))

    assert len(df) == total, "❌ Има дублиране или липсващи редове"

In [4]:
def test_show_duplicates():
    df = merge_tables()

    duplicates = df[df.duplicated(subset=['id'], keep=False)]

    print("Duplicate rows:", len(duplicates))
    print("Unique duplicated transactions:", duplicates['id'].nunique())

    return duplicates.sort_values('id')

In [5]:
def test_nulls():
    df = merge_tables()

    critical_cols = ['id', 'card_number', 'material', 'date', 'final_price']

    nulls = df[critical_cols].isnull().sum()

    print(nulls)

    assert nulls.sum() == 0, "❌ Има NULL в критични колони"

In [6]:
def test_negative_values():
    df = merge_tables()

    assert (df['final_price'] >= 0).all(), "❌ Има отрицателни final_price"
    assert (df['price'] >= 0).all(), "❌ Има отрицателни price"
    assert (df['bill_qty'] >= 0).all(), "❌ Има отрицателни количества"

In [7]:
def test_price_logic():
    df = merge_tables()

    wrong = df[
        (df['discount'].notnull()) &
        (df['discount'] > 0) &
        (df['final_price'] > df['price'])
    ]

    print("Wrong discount rows:", len(wrong))

    assert len(wrong) == 0

In [8]:
def test_price_coverage():
    df = merge_tables()

    missing = df[df['base_price'].isnull()]

    print("Missing price matches:", len(missing))

    # не assert, само мониторинг

In [9]:
def test_price_duplicates():
    import sqlite3
    import pandas as pd

    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT eik, product, date, COUNT(*) as cnt
    FROM prices
    GROUP BY eik, product, date
    HAVING cnt > 1
    """

    df = pd.read_sql(query, conn)
    conn.close()

    print("Duplicate price rows:", len(df))

    return df

In [10]:
def test_date_logic():
    df = merge_tables()

    wrong = df[df['date'] < '2000-01-01']

    assert len(wrong) == 0, "❌ Невалидни дати"

In [11]:
def test_rounding():
    df = merge_tables()

    decimals = df['final_price'].apply(lambda x: round(x, 2) == x)

    assert decimals.all(), "❌ Има стойности с повече от 2 десетични"

In [12]:
def run_all_tests():
    import pandas as pd
    import sqlite3
    from datetime import datetime

    results = []

    def add_result(test_name, status, details=""):
        results.append({
            "test": test_name,
            "status": status,
            "details": details
        })

    # =========================
    # LOAD DATA
    # =========================
    try:
        df = merge_tables()
        add_result("load_data", "PASS", f"Rows: {len(df)}")
    except Exception as e:
        add_result("load_data", "FAIL", str(e))
        return pd.DataFrame(results)

    # =========================
    # TEST 1: ROW COUNT
    # =========================
    try:
        conn = sqlite3.connect(DB_PATH)
        total = conn.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
        conn.close()

        if len(df) == total:
            add_result("row_count", "PASS", f"{len(df)} rows")
        else:
            add_result("row_count", "FAIL", f"Expected {total}, got {len(df)}")
    except Exception as e:
        add_result("row_count", "FAIL", str(e))

    # =========================
    # TEST 2: DUPLICATES
    # =========================
    try:
        duplicates = df[df.duplicated(subset=['id'], keep=False)]
        if len(duplicates) == 0:
            add_result("duplicates", "PASS")
        else:
            add_result("duplicates", "FAIL", f"{len(duplicates)} duplicate rows")
    except Exception as e:
        add_result("duplicates", "FAIL", str(e))

    # =========================
    # TEST 3: NULL CHECK
    # =========================
    try:
        critical_cols = ['id', 'card_number', 'material', 'date', 'final_price']
        nulls = df[critical_cols].isnull().sum().sum()

        if nulls == 0:
            add_result("nulls", "PASS")
        else:
            add_result("nulls", "FAIL", f"{nulls} null values")
    except Exception as e:
        add_result("nulls", "FAIL", str(e))

    # =========================
    # TEST 4: NEGATIVE VALUES
    # =========================
    try:
        if (df['final_price'] < 0).any():
            add_result("negative_price", "FAIL")
        else:
            add_result("negative_price", "PASS")
    except Exception as e:
        add_result("negative_price", "FAIL", str(e))

    # =========================
    # TEST 5: PRICE LOGIC (DISCOUNT ONLY)
    # =========================
    try:
        wrong = df[
            (df['discount'].notnull()) &
            (df['discount'] > 0) &
            (df['final_price'] > df['price'])
        ]

        if len(wrong) == 0:
            add_result("price_logic", "PASS")
        else:
            add_result("price_logic", "FAIL", f"{len(wrong)} discount logic errors")
    except Exception as e:
        add_result("price_logic", "FAIL", str(e))

    # =========================
    # TEST 6: ROUNDING
    # =========================
    try:
        decimals = df['final_price'].apply(lambda x: round(x, 2) == x)
        if decimals.all():
            add_result("rounding", "PASS")
        else:
            add_result("rounding", "FAIL")
    except Exception as e:
        add_result("rounding", "FAIL", str(e))


    # =========================
    # FINAL REPORT
    # =========================
    report = pd.DataFrame(results)

    print("\n===== QA REPORT =====")
    print(report)

    # =========================
    # SAVE TO EXCEL
    # =========================
    filename = f"qa_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    # report.to_excel(filename, index=False)

    print(f"\nSaved report: {filename}")

    return report

In [13]:
run_all_tests()


===== QA REPORT =====
             test status    details
0       load_data   PASS  Rows: 830
1       row_count   PASS   830 rows
2      duplicates   PASS           
3           nulls   PASS           
4  negative_price   PASS           
5     price_logic   PASS           
6        rounding   PASS           

Saved report: qa_report_20260618_101940.xlsx


,test,status,details
0,load_data,PASS,Rows: 830
1,row_count,PASS,830 rows
2,duplicates,PASS,
3,nulls,PASS,
4,negative_price,PASS,
5,price_logic,PASS,
6,rounding,PASS,
